In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np

In [ ]:
def frec_table(data: dict, s: str):
    """
    Tabla que muestra las frecuencias de los caracteres de la lengua española en un diccionario.
    
    data: Contenido de las transcripciones.
    s: Datos específicos con los que se está trabajando. Ej.: 'Datos Entrenamiento'
    """

    # 1. Process real count
    text = "".join(str(value) for value in data.values())
    real_count = Counter(text)

    # 2. Define groups
    lowercase = "abcdefghijklmnñopqrstuvwxyz"
    uppercase = lowercase.upper()
    all_accents = "áéíóúüÁÉÍÓÚÜ"
    signs = ".,;:¡!¿?()[]-@#&*/\"\'"

    max_len = max(len(lowercase), len(uppercase), len(all_accents), len(signs))
    
    table_data = []
    freq_values = []  # Para guardar todas las frecuencias numéricas

    for i in range(max_len):
        fila = []
        for group in [lowercase, uppercase, all_accents, signs]:
            char = group[i] if i < len(group) else ""
            freq = real_count[char] if char != "" else ""
            fila.extend([char, freq])
            if isinstance(freq, int):
                freq_values.append(freq)
        table_data.append(fila)

    columns = ["Min", "Freq", "May", "Freq", "Tildes", "Freq", "Signos", "Freq"]
    df = pd.DataFrame(table_data, columns=columns)

    # 3. Calcular umbrales por cuartiles
    if freq_values:
        q1, q2, q3 = np.percentile(freq_values, [25, 50, 75])
    else:
        q1 = q2 = q3 = 0

    # 4. Crear figura
    fig, ax = plt.subplots(figsize=(14, 12))
    ax.axis('off')
    
    table = ax.table(
        cellText=df.values, 
        colLabels=df.columns, 
        cellLoc='center', 
        loc='center'
    )

    # 5. Aplicar colores por frecuencia relativa
    for (row, col), cell in table.get_celld().items():
        
        if row == 0:
            cell.set_text_props(weight='bold', color='white')
            cell.set_facecolor('#2c3e50')
            continue

        if col in [1, 3, 5, 7]:
            value = cell.get_text().get_text()

            if value == "":
                cell.set_facecolor('#f2f2f2')
                continue

            value = int(value)

            # Asignación por cuartiles
            if value >= q3:
                cell.set_facecolor('#2ecc71')  # Verde (frecuencia alta)
                cell.set_text_props(color='white', weight='bold')
            elif value >= q2:
                cell.set_facecolor('#f1c40f')  # Amarillo
                cell.set_text_props(color='black')
            elif value >= q1:
                cell.set_facecolor('#e67e22')  # Naranja
                cell.set_text_props(color='white')
            else:
                cell.set_facecolor('#e74c3c')  # Rojo (frecuencia baja)
                cell.set_text_props(color='white')

        else:
            cell.set_facecolor('#ffffff')

    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.8)

    plt.title(f"Análisis de Caracteres: Frecuencias por Intensidad ({s})", fontsize=18, pad=30)
    plt.show()

In [ ]:
with open('synthetic_manuscript/labels.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

frec_table(data, "Dataset sintético")

In [ ]:
with open('IAM_img/train/transcriptions.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

frec_table(data, "Train-IAM")

In [ ]:
with open('final_dataset/train.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

frec_table(data, "Train-Datos Reales")

In [ ]:
import os
import cv2
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt

IMAGE_FOLDER = "dataset1/datos_ent_augmented"

heights, widths, aspectos = [], [], []

for filename in os.listdir(IMAGE_FOLDER):
    if not filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        continue
    img = cv2.imread(os.path.join(IMAGE_FOLDER, filename), cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue
    h, w = img.shape
    heights.append(h)
    widths.append(w)
    aspectos.append(w / h)

heights  = np.array(heights)
widths   = np.array(widths)
aspectos = np.array(aspectos)

print("═" * 45)
print("  ALTURAS (px)")
print("═" * 45)
print(f"  Min:        {heights.min()}")
print(f"  Max:        {heights.max()}")
print(f"  Media:      {heights.mean():.1f}")
print(f"  Mediana:    {np.median(heights):.1f}")
print(f"  P90:        {np.percentile(heights, 90):.1f}")
print(f"  P95:        {np.percentile(heights, 95):.1f}")

print("\n" + "═" * 45)
print("  ANCHURAS (px)")
print("═" * 45)
print(f"  Min:        {widths.min()}")
print(f"  Max:        {widths.max()}")
print(f"  Media:      {widths.mean():.1f}")
print(f"  Mediana:    {np.median(widths):.1f}")
print(f"  P90:        {np.percentile(widths, 90):.1f}")
print(f"  P95:        {np.percentile(widths, 95):.1f}")

print("\n" + "═" * 45)
print("  RATIO ASPECTO (ancho/alto)")
print("═" * 45)
print(f"  Min:        {aspectos.min():.2f}")
print(f"  Max:        {aspectos.max():.2f}")
print(f"  Media:      {aspectos.mean():.2f}")
print(f"  Mediana:    {np.median(aspectos):.2f}")

# Cuántas imágenes se perderían con cada IMG_WIDTH
print("\n" + "═" * 45)
print("  % IMÁGENES QUE CABEN EN CADA IMG_WIDTH")
print("═" * 45)
for w_limit in [256, 512, 768, 1024, 1280, 1600, 2048]:
    pct = (widths <= w_limit).mean() * 100
    print(f"  IMG_WIDTH={w_limit:5d}: {pct:5.1f}% caben completas")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(heights, bins=50, color='steelblue')
axes[0].axvline(np.percentile(heights, 95), color='red', linestyle='--', label='P95')
axes[0].set_title("Distribución de alturas")
axes[0].set_xlabel("px"); axes[0].legend()

axes[1].hist(widths, bins=50, color='steelblue')
axes[1].axvline(np.percentile(widths, 95), color='red', linestyle='--', label='P95')
axes[1].set_title("Distribución de anchuras")
axes[1].set_xlabel("px"); axes[1].legend()

axes[2].hist(aspectos, bins=50, color='steelblue')
axes[2].set_title("Ratio aspecto (ancho/alto)")
axes[2].set_xlabel("w/h")

plt.tight_layout()
plt.savefig("distribucion_tamaños.png", dpi=150)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from PIL import Image
import io

df = pd.read_parquet("IAM/data/train.parquet")

widths, heights = [], []

for _, row in df.iterrows():
    img_bytes = row['image']['bytes']  # ajusta el nombre
    img = Image.open(io.BytesIO(img_bytes))
    w, h = img.size
    widths.append(w)
    heights.append(h)

widths  = np.array(widths)
heights = np.array(heights)

print(f"Ancho  - min: {widths.min()} max: {widths.max()} promedio: {widths.mean():.0f}")
print(f"Alto   - min: {heights.min()} max: {heights.max()} promedio: {heights.mean():.0f}")

for p in [50, 75, 90, 95, 99]:
    print(f"  P{p} ancho: {np.percentile(widths, p):.0f}px")

for w_limit in [1024, 1280, 1536, 2048, 2560, 3072]:
    pct = (widths <= w_limit).mean() * 100
    print(f"  IMG_WIDTH={w_limit}: {pct:.1f}% caben completas")

Pre

IMG_HEIGHT = 128

IMG_WIDTH  = 768

BATCH_SIZE = 32

FineTuning

IMG_HEIGHT = 128

IMG_WIDTH  = 3072   # cubre el 96.2%, el 3.8% restante se recorta

BATCH_SIZE = 8      # imagen 4x más ancha → 4x más memoria